# Professional Practice X1: Clustering Algorithm Comparison

## Systematic Evaluation Framework

**Production Challenge**: You have a new dataset. Which clustering algorithm should you choose?

**Goal**: Build a systematic comparison framework to evaluate K-Means, Hierarchical, DBSCAN, and GMM on different data patterns.

**Key Question**: How do algorithms perform on blob-shaped vs. non-convex clusters?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print('✅ Loaded!')

## Step 1: Create Diverse Test Datasets

Different data patterns challenge clustering algorithms differently.

In [ ]:
# Create datasets with different properties
datasets = {}

# 1. Well-separated blobs (easy)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)
datasets['Blobs (easy)'] = (X_blobs, y_blobs, 4)

# 2. Overlapping blobs (moderate)
X_overlap, y_overlap = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)
datasets['Overlapping'] = (X_overlap, y_overlap, 3)

# 3. Non-convex moons (hard for K-Means)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
datasets['Moons (non-convex)'] = (X_moons, y_moons, 2)

# 4. Concentric circles (hard for K-Means)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
datasets['Circles (concentric)'] = (X_circles, y_circles, 2)

# 5. Varying density
X_density = np.vstack([
    np.random.randn(100, 2) * 0.3,
    np.random.randn(50, 2) * 0.1 + [3, 3],
    np.random.randn(150, 2) * 0.6 + [0, 4]
])
y_density = np.hstack([np.zeros(100), np.ones(50), np.full(150, 2)])
datasets['Varying Density'] = (X_density, y_density, 3)

# Visualize all datasets
fig, axes = plt.subplots(1, len(datasets), figsize=(18, 3))
for ax, (name, (X, y, _)) in zip(axes, datasets.items()):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', s=20, alpha=0.6)
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Test Datasets', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print(f'✅ Created {len(datasets)} test datasets')

## Step 2: Define Algorithms

Configure each algorithm with reasonable defaults.

In [ ]:
def get_algorithms(n_clusters):
    """Return configured clustering algorithms"""
    return {
        'K-Means': KMeans(n_clusters=n_clusters, random_state=42, n_init=10),
        'Hierarchical (Ward)': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
        'Hierarchical (Average)': AgglomerativeClustering(n_clusters=n_clusters, linkage='average'),
        'DBSCAN': DBSCAN(eps=0.3, min_samples=5),
        'GMM': GaussianMixture(n_components=n_clusters, random_state=42, covariance_type='full')
    }

# Test on one dataset
algorithms = get_algorithms(3)
print(f'✅ Configured {len(algorithms)} algorithms')
for name in algorithms.keys():
    print(f'   - {name}')

## Step 3: Run Systematic Comparison

Evaluate each algorithm on each dataset with multiple metrics.

In [ ]:
# Comparison framework
results = []

for ds_name, (X, y_true, n_clusters) in datasets.items():
    # Standardize (important for distance-based methods)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    algorithms = get_algorithms(n_clusters)
    
    for algo_name, algo in algorithms.items():
        try:
            # Fit and predict
            if hasattr(algo, 'fit_predict'):
                labels = algo.fit_predict(X_scaled)
            else:
                labels = algo.fit(X_scaled).predict(X_scaled)
            
            # Skip if only one cluster found
            n_labels = len(set(labels)) - (1 if -1 in labels else 0)
            if n_labels < 2:
                continue
            
            # Compute metrics (internal validation - no ground truth needed)
            sil = silhouette_score(X_scaled, labels)
            db = davies_bouldin_score(X_scaled, labels)
            ch = calinski_harabasz_score(X_scaled, labels)
            
            results.append({
                'Dataset': ds_name,
                'Algorithm': algo_name,
                'Silhouette': sil,
                'Davies-Bouldin': db,
                'Calinski-Harabasz': ch,
                'N_Clusters': n_labels
            })
            
        except Exception as e:
            print(f"Error with {algo_name} on {ds_name}: {e}")

import pandas as pd
results_df = pd.DataFrame(results)
print('\n✅ Comparison complete!')
print(f'Total evaluations: {len(results)}')

## Step 4: Analyze Results

Which algorithm wins on each dataset?

In [ ]:
# Find best algorithm per dataset (by Silhouette score)
print("=== BEST ALGORITHM PER DATASET ===\n")

for ds_name in datasets.keys():
    ds_results = results_df[results_df['Dataset'] == ds_name]
    best = ds_results.loc[ds_results['Silhouette'].idxmax()]
    
    print(f"{ds_name}:")
    print(f"  Winner: {best['Algorithm']}")
    print(f"  Silhouette: {best['Silhouette']:.3f}")
    print(f"  Davies-Bouldin: {best['Davies-Bouldin']:.3f} (lower is better)")
    print(f"  Calinski-Harabasz: {best['Calinski-Harabasz']:.1f} (higher is better)")
    print()

print('✅ Analysis complete!')

## Step 5: Visualize Comparison

In [ ]:
# Heatmap of Silhouette scores
pivot = results_df.pivot(index='Algorithm', columns='Dataset', values='Silhouette')

plt.figure(figsize=(12, 6))
im = plt.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=1.0)
plt.colorbar(im, label='Silhouette Score')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha='right')
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel('Dataset', fontsize=12)
plt.ylabel('Algorithm', fontsize=12)
plt.title('Clustering Performance Comparison (Silhouette Score)', 
         fontweight='bold', fontsize=14)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        value = pivot.iloc[i, j]
        if not np.isnan(value):
            color = 'white' if value < 0.3 else 'black'
            plt.text(j, i, f'{value:.2f}', ha='center', va='center', 
                    color=color, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ Heatmap complete!')

## Step 6: Visual Clustering Results

In [ ]:
# Show clustering results for selected dataset
selected_dataset = 'Moons (non-convex)'
X, y_true, n_clusters = datasets[selected_dataset]
X_scaled = scaler.fit_transform(X)

algorithms = get_algorithms(n_clusters)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Ground truth
axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='tab10', s=30, alpha=0.7, edgecolors='black', linewidths=0.5)
axes[0].set_title('Ground Truth', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Algorithm results
for idx, (algo_name, algo) in enumerate(algorithms.items(), 1):
    try:
        if hasattr(algo, 'fit_predict'):
            labels = algo.fit_predict(X_scaled)
        else:
            labels = algo.fit(X_scaled).predict(X_scaled)
        
        # Compute score
        if len(set(labels)) > 1:
            score = silhouette_score(X_scaled, labels)
            title = f'{algo_name}\nSil: {score:.3f}'
        else:
            title = f'{algo_name}\nFailed'
        
        axes[idx].scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=30, alpha=0.7, edgecolors='black', linewidths=0.5)
        axes[idx].set_title(title, fontweight='bold', fontsize=11)
        axes[idx].grid(True, alpha=0.3)
    except:
        axes[idx].set_title(f'{algo_name}\nError', fontweight='bold')

plt.suptitle(f'Clustering Results: {selected_dataset}', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('✅ Visual comparison complete!')

## Production Decision Framework

### Algorithm Selection Guide

| Pattern | Best Algorithm | Why |
|---------|---------------|-----|
| **Well-separated blobs** | K-Means, GMM | Fast, assumes spherical clusters |
| **Overlapping clusters** | GMM | Soft assignment handles uncertainty |
| **Non-convex shapes** | DBSCAN, Hierarchical | Density/linkage-based, no shape assumption |
| **Varying density** | Hierarchical, DBSCAN | Adapt to local density |
| **Large datasets** | K-Means, Mini-Batch K-Means | O(n) complexity |

### Metric Interpretation

- **Silhouette** (higher = better): Measures how similar samples are to their own cluster vs. other clusters
  - > 0.7: Strong structure
  - 0.5-0.7: Reasonable
  - < 0.5: Weak/overlapping
  
- **Davies-Bouldin** (lower = better): Ratio of within-cluster to between-cluster distances
  
- **Calinski-Harabasz** (higher = better): Ratio of between-cluster variance to within-cluster variance

### Production Checklist

✅ **Preprocess**: Always standardize features for distance-based methods  
✅ **Validate**: Use multiple metrics (Silhouette, DB, CH)  
✅ **Visualize**: Plot results when possible (2D/3D projection)  
✅ **Tune**: Grid search over n_clusters, eps, linkage  
✅ **Ensemble**: Combine multiple algorithms for robustness  
✅ **Monitor**: Track cluster quality metrics over time